# ORION wake word training - openWakeWord Colab

Runtime recomendado: Google Colab 2025.07, Python 3.11, GPU T4.

Este notebook entrena un modelo ONNX temporal `orion_v2.onnx` para la palabra de activacion futura de ORION. Todo lo persistente se guarda en Google Drive bajo `/content/drive/MyDrive/orion_wakeword_training/`.

No modifica el pipeline estable de ORION, no usa AudioSet, no convierte a TFLite y no guarda artefactos pesados en Git.

In [ ]:
# 1. Montar Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/orion_wakeword_training')
REPOS_DIR = BASE_DIR / 'repos'
DATASETS_DIR = BASE_DIR / 'datasets'
CLIPS_DIR = BASE_DIR / 'clips'
FEATURES_DIR = BASE_DIR / 'features'
MODELS_DIR = BASE_DIR / 'models'
EXPORT_DIR = BASE_DIR / 'exports'
for path in [BASE_DIR, REPOS_DIR, DATASETS_DIR, CLIPS_DIR, FEATURES_DIR, MODELS_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('Directorio persistente:', BASE_DIR)

In [ ]:
# 2. Verificar Python/GPU
import os, platform, subprocess, sys
print('Python:', sys.version)
print('Platform:', platform.platform())
!nvidia-smi
assert sys.version_info[:2] == (3, 11), 'Usa runtime Colab Python 3.11'
gpu_name = subprocess.check_output('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, text=True).strip()
print('GPU:', gpu_name)
assert 'T4' in gpu_name, 'Selecciona GPU T4 en Runtime > Change runtime type'

In [ ]:
# 3. Instalar dependencias necesarias en una sola celda
# No instalar tensorflow-cpu==2.8.1. No se convierte a TFLite.
%pip install -q --upgrade pip
%pip install -q \
  piper-phonemize==1.1.0 \
  webrtcvad \
  torchinfo==1.8.0 \
  torchmetrics==1.2.0 \
  lightning-utilities \
  speechbrain==0.5.14 \
  audiomentations==0.33.0 \
  torch-audiomentations==0.11.0 \
  acoustics==0.2.6 \
  pronouncing==0.2.0 \
  deep-phonemizer==0.0.19 \
  mutagen==1.47.0 \
  librosa==0.10.2.post1 \
  soxr==0.5.0.post1 \
  onnxruntime==1.27.0 \
  onnx==1.16.1 \
  ai-edge-litert==2.1.5 \
  speexdsp-ns==0.1.2 \
  fsspec==2023.6.0 \
  datasets==2.14.4

import importlib.metadata as md
for pkg in ['piper-phonemize', 'speechbrain', 'librosa', 'soxr', 'onnxruntime', 'onnx', 'datasets', 'fsspec']:
    print(pkg, md.version(pkg))

In [ ]:
# 4. Clonar repositorios compatibles
import subprocess, os
OPENWAKEWORD_DIR = REPOS_DIR / 'openWakeWord'
PIPER_DIR = REPOS_DIR / 'piper-sample-generator'

def run(cmd, cwd=None, env=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, check=True)

if not OPENWAKEWORD_DIR.exists():
    run(['git', 'clone', 'https://github.com/dscripka/openWakeWord.git', OPENWAKEWORD_DIR])
else:
    print('Reutilizando', OPENWAKEWORD_DIR)

if not PIPER_DIR.exists():
    run(['git', 'clone', 'https://github.com/rhasspy/piper-sample-generator.git', PIPER_DIR])
run(['git', 'fetch', '--tags'], cwd=PIPER_DIR)
run(['git', 'checkout', 'v2.0.0'], cwd=PIPER_DIR)

os.environ['PYTHONPATH'] = f"{OPENWAKEWORD_DIR}:{PIPER_DIR}:" + os.environ.get('PYTHONPATH', '')
print('PYTHONPATH=', os.environ['PYTHONPATH'])

In [ ]:
# 5. Aplicar parches de compatibilidad
import importlib.util, re, site

def patch_torch_load_file(path):
    path = Path(path)
    text = path.read_text(encoding='utf-8')
    def repl(match):
        args = match.group(1)
        if 'weights_only' in args:
            return match.group(0)
        return f"torch.load({args}, weights_only=False)"
    patched = re.sub(r"torch\.load\(([^\n)]*)\)", repl, text)
    if patched != text:
        path.write_text(patched, encoding='utf-8')
        print('Patched torch.load:', path)

patch_torch_load_file(PIPER_DIR / 'generate_samples.py')

spec = importlib.util.find_spec('dp') or importlib.util.find_spec('deep_phonemizer')
if spec and spec.origin:
    dp_root = Path(spec.origin).resolve().parent
    for py_file in dp_root.rglob('*.py'):
        if 'torch.load(' in py_file.read_text(encoding='utf-8', errors='ignore'):
            patch_torch_load_file(py_file)
else:
    print('deep-phonemizer package not found for patch scan')

# openWakeWord 0.6 train.py exporta ONNX y luego intenta TFLite. Esta fase es ONNX-only.
train_py = OPENWAKEWORD_DIR / 'openwakeword' / 'train.py'
text = train_py.read_text(encoding='utf-8')
text = text.replace(
    "        # Convert the model from onnx to tflite format\n        convert_onnx_to_tflite(os.path.join(config[\"output_dir\"], config[\"model_name\"] + \".onnx\"),\n                               os.path.join(config[\"output_dir\"], config[\"model_name\"] + \".tflite\"))",
    "        # ORION patch: ONNX-only training. Do not convert to TFLite.\n        logging.info('ORION patch: skipping TFLite conversion; ONNX export only.')"
)
train_py.write_text(text, encoding='utf-8')
print('Parches aplicados')

In [ ]:
# 6. Descargar modelos auxiliares y datasets
import urllib.request, zipfile, tarfile, shutil

def download(url, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print('Reutilizando', dest)
        return dest
    print('Descargando', url, '->', dest)
    urllib.request.urlretrieve(url, dest)
    return dest

OWW_RESOURCES = OPENWAKEWORD_DIR / 'openwakeword' / 'resources' / 'models'
OWW_RESOURCES.mkdir(parents=True, exist_ok=True)
BASE_RELEASE = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
download(f'{BASE_RELEASE}/melspectrogram.onnx', OWW_RESOURCES / 'melspectrogram.onnx')
download(f'{BASE_RELEASE}/embedding_model.onnx', OWW_RESOURCES / 'embedding_model.onnx')

PIPER_VOICE_DIR = MODELS_DIR / 'piper'
PIPER_VOICE_DIR.mkdir(parents=True, exist_ok=True)
PIPER_BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium'
PIPER_MODEL = download(f'{PIPER_BASE}/en_US-libritts_r-medium.pt', PIPER_VOICE_DIR / 'en_US-libritts_r-medium.pt')
download(f'{PIPER_BASE}/en_US-libritts_r-medium.onnx.json', PIPER_VOICE_DIR / 'en_US-libritts_r-medium.onnx.json')

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
download(f'{BASE_RELEASE}/openwakeword_features_ACAV100M_2000_hrs_16bit.npy', FEATURES_DIR / 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy')
download(f'{BASE_RELEASE}/validation_set_features.npy', FEATURES_DIR / 'validation_set_features.npy')

# Fondos: FMA y MIT RIR. No usar AudioSet.
FMA_PATH = DATASETS_DIR / 'fma'
MIT_RIRS_PATH = DATASETS_DIR / 'mit_rirs'
FMA_ZIP = download('https://os.unil.cloud.switch.ch/fma/fma_small.zip', DATASETS_DIR / 'fma_small.zip')
if not FMA_PATH.exists():
    FMA_PATH.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(FMA_ZIP) as zf:
        zf.extractall(FMA_PATH)

MIT_RIRS_ZIP = download('https://mcdermottlab.mit.edu/Reverb/IRMAudio/Audio.zip', DATASETS_DIR / 'mit_rirs_audio.zip')
if not MIT_RIRS_PATH.exists():
    MIT_RIRS_PATH.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(MIT_RIRS_ZIP) as zf:
        zf.extractall(MIT_RIRS_PATH)

print('FMA_PATH=', FMA_PATH)
print('MIT_RIRS_PATH=', MIT_RIRS_PATH)

In [ ]:
# 7. Crear my_model_v2.yaml
import yaml

target_phrase = ['oh ree on', 'o ree on', 'oree on', 'orion']
model_name = 'orion_v2'
n_samples = 1500
n_samples_val = 1000
steps = 10000
target_accuracy = 0.6
target_recall = 0.25
background_paths = [str(FMA_PATH), str(MIT_RIRS_PATH)]

CONFIG_PATH = BASE_DIR / 'my_model_v2.yaml'
config = {
    'target_phrase': target_phrase,
    'model_name': model_name,
    'output_dir': str(BASE_DIR / 'training_output'),
    'piper_sample_generator_path': str(PIPER_DIR),
    'n_samples': n_samples,
    'n_samples_val': n_samples_val,
    'steps': steps,
    'target_accuracy': target_accuracy,
    'target_recall': target_recall,
    'background_paths': background_paths,
    'background_paths_duplication_rate': [1 for _ in background_paths],
    'rir_paths': [str(MIT_RIRS_PATH)],
    'custom_negative_phrases': ['orient', 'oreo', 'onion', 'or i am', 'area on', 'oh really', 'turn on', 'online'],
    'tts_batch_size': 16,
    'augmentation_rounds': 1,
    'augmentation_batch_size': 16,
    'feature_data_files': {
        'negative': str(FEATURES_DIR / 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy'),
    },
    'false_positive_validation_data_path': str(FEATURES_DIR / 'validation_set_features.npy'),
    'model_type': 'dnn',
    'layer_size': 64,
    'batch_n_per_class': 64,
    'max_negative_weight': 1500,
    'target_false_positives_per_hour': 0.5,
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(CONFIG_PATH)
print(CONFIG_PATH.read_text())

In [ ]:
# 8. Generar clips sinteticos (idempotente)
env = os.environ.copy()
env['PYTHONPATH'] = f"{OPENWAKEWORD_DIR}:{PIPER_DIR}:" + env.get('PYTHONPATH', '')
TRAIN_PY = OPENWAKEWORD_DIR / 'openwakeword' / 'train.py'
OUTPUT_MODEL_DIR = BASE_DIR / 'training_output' / model_name
expected_dirs = ['positive_train', 'positive_test', 'negative_train', 'negative_test']
if all((OUTPUT_MODEL_DIR / d).exists() and len(list((OUTPUT_MODEL_DIR / d).glob('*.wav'))) >= (n_samples if 'train' in d else n_samples_val) for d in expected_dirs):
    print('Clips existentes suficientes; se reutilizan.')
else:
    run([sys.executable, TRAIN_PY, '--training_config', CONFIG_PATH, '--generate_clips'], env=env)
print('Clips en', OUTPUT_MODEL_DIR)

In [ ]:
# 9. Verificar conteos
def count_wavs(path):
    return len(list(Path(path).glob('*.wav')))

counts = {
    'positive_train': count_wavs(OUTPUT_MODEL_DIR / 'positive_train'),
    'positive_test': count_wavs(OUTPUT_MODEL_DIR / 'positive_test'),
    'negative_train': count_wavs(OUTPUT_MODEL_DIR / 'negative_train'),
    'negative_test': count_wavs(OUTPUT_MODEL_DIR / 'negative_test'),
}
print(counts)
assert counts['positive_train'] >= 1500
assert counts['positive_test'] >= 1000
assert counts['negative_train'] >= 1500
assert counts['negative_test'] >= 1000

In [ ]:
# 10. Aumentar clips y generar features (idempotente)
feature_files = [
    OUTPUT_MODEL_DIR / 'positive_features_train.npy',
    OUTPUT_MODEL_DIR / 'positive_features_test.npy',
    OUTPUT_MODEL_DIR / 'negative_features_train.npy',
    OUTPUT_MODEL_DIR / 'negative_features_test.npy',
]
if all(path.exists() and path.stat().st_size > 0 for path in feature_files):
    print('Features existentes; se reutilizan.')
else:
    run([sys.executable, TRAIN_PY, '--training_config', CONFIG_PATH, '--augment_clips'], env=env)
for path in feature_files:
    print(path, path.exists(), path.stat().st_size if path.exists() else 0)

In [ ]:
# 11. Entrenar modelo ONNX
ONNX_PATH = BASE_DIR / 'training_output' / f'{model_name}.onnx'
if ONNX_PATH.exists() and ONNX_PATH.stat().st_size > 0:
    print('Modelo ONNX existente; se reutiliza:', ONNX_PATH)
else:
    run([sys.executable, TRAIN_PY, '--training_config', CONFIG_PATH, '--train_model'], env=env)
print('ONNX_PATH=', ONNX_PATH)

In [ ]:
# 12. Verificar que exista orion_v2.onnx
assert ONNX_PATH.exists(), f'No existe {ONNX_PATH}'
assert ONNX_PATH.stat().st_size > 0, f'Modelo vacio: {ONNX_PATH}'
print('OK:', ONNX_PATH, ONNX_PATH.stat().st_size, 'bytes')

In [ ]:
# 13. Copiar a carpeta persistente en Drive
FINAL_MODEL = EXPORT_DIR / 'orion_v2.onnx'
shutil.copy2(ONNX_PATH, FINAL_MODEL)
print('Modelo persistente:', FINAL_MODEL)

In [ ]:
# 14. Descargar modelo desde Colab
from google.colab import files
files.download(str(FINAL_MODEL))